# Perbandingan Model - Prediksi Student Dropout

Notebook ini membandingkan **5 model klasifikasi** untuk prediksi dropout mahasiswa:
- Logistic Regression
- Random Forest
- XGBoost
- SVM
- Neural Network (MLP)

Metrik yang dibandingkan: **Accuracy, Precision, Recall, F1-Score, ROC-AUC**

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

print("Libraries loaded successfully ✓")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

---
## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../student_dropout_dataset_v3.csv')

print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nDistribusi target (Dropout):")
print(df['Dropout'].value_counts())
print(f"\nPersentase Dropout: {df['Dropout'].mean()*100:.2f}%")
df.head(3)

---
## 3. Data Preprocessing

In [ ]:
target_col = 'Dropout'
drop_cols  = ['Student_ID']

# Label Encode semua kolom kategorik
df_proc = df.drop(columns=drop_cols).copy()
categorical_cols = df_proc.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in categorical_cols:
    df_proc[col] = le.fit_transform(df_proc[col].astype(str))

X = df_proc.drop(columns=[target_col])
y = df_proc[target_col]

# Split 80/20 stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Imputasi + Scaling
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

X_train_s = scaler.fit_transform(X_train_imp)
X_test_s  = scaler.transform(X_test_imp)

print(f"Train : {X_train_s.shape[0]} sampel")
print(f"Test  : {X_test_s.shape[0]} sampel")
print(f"Fitur : {X_train_s.shape[1]}")

---
## 4. Train & Evaluate Semua Model

Kita melatih 5 model dengan setting default yang wajar, lalu mengevaluasi masing-masing.

In [ ]:
import time

# Hitung class weight untuk handle imbalanced
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

# Definisikan semua model
models = {
    "Logistic Regression": LogisticRegression(
        C=1, solver='liblinear', class_weight='balanced',
        max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced',
        n_jobs=-1, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic', random_state=42,
        verbosity=0, n_jobs=-1
    ),
    "SVM": SVC(
        kernel='rbf', C=1, gamma='scale',
        class_weight='balanced', probability=True, random_state=42
    ),
    "Neural Network": MLPClassifier(
        hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
        alpha=0.001, max_iter=300, early_stopping=True,
        random_state=42, verbose=False
    ),
}

# ──────────────────────────────────────────────────────
# Train & evaluate setiap model
# ──────────────────────────────────────────────────────
results       = []
model_objects = {}
roc_data      = {}   # simpan (fpr, tpr, auc) untuk plot

print(f"{'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7} {'Time':>7}")
print("─" * 70)

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train_s, y_train)
    elapsed = time.time() - t0

    y_pred  = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]

    acc  = accuracy_score (y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score   (y_test, y_pred)
    f1   = f1_score       (y_test, y_pred)
    auc  = roc_auc_score  (y_test, y_proba)

    results.append({
        "Model"    : name,
        "Accuracy" : acc,
        "Precision": prec,
        "Recall"   : rec,
        "F1-Score" : f1,
        "ROC-AUC"  : auc,
    })
    model_objects[name] = (model, y_pred, y_proba)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data[name] = (fpr, tpr, auc)

    print(f"{name:<22} {acc:>7.4f} {prec:>7.4f} {rec:>7.4f} {f1:>7.4f} {auc:>7.4f} {elapsed:>5.1f}s")

print("─" * 70)

---
## 5. Tabel Perbandingan Metrik

In [ ]:
df_results = pd.DataFrame(results).set_index("Model")
metric_cols = ["Accuracy","Precision","Recall","F1-Score","ROC-AUC"]

# Highlight best value per metric
def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: #b7f7c4; font-weight: bold' if v else '' for v in is_max]

styled = (
    df_results[metric_cols]
    .style
    .apply(highlight_max)
    .format("{:.4f}")
    .set_caption("Perbandingan Metrik Evaluasi — Student Dropout Classification")
)
styled

---
## 6. Confusion Matrix — Semua Model

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle("Confusion Matrix — Semua Model", fontsize=14, fontweight='bold', y=1.02)

for ax, (name, (_, y_pred, _)) in zip(axes, model_objects.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Tdk Dropout', 'Dropout'],
        yticklabels=['Tdk Dropout', 'Dropout'],
        annot_kws={"size": 11}
    )
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Prediksi', fontsize=9)
    ax.set_ylabel('Aktual',   fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Visualisasi Perbandingan

In [ ]:
colors = ['#e55', '#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── ROC Curve semua model di satu plot ──────────────────
ax = axes[0]
for (name, (fpr, tpr, auc)), color in zip(roc_data.items(), colors):
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={auc:.3f})")
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.set_title('ROC Curve — Semua Model', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)

# ── Bar chart F1-Score & ROC-AUC ─────────────────────────
ax2 = axes[1]
x  = np.arange(len(df_results))
w  = 0.35
f1_vals  = df_results['F1-Score'].values
auc_vals = df_results['ROC-AUC'].values

bars1 = ax2.bar(x - w/2, f1_vals,  w, label='F1-Score', color='steelblue',  alpha=0.85)
bars2 = ax2.bar(x + w/2, auc_vals, w, label='ROC-AUC',  color='darkorange', alpha=0.85)

ax2.set_xticks(x)
ax2.set_xticklabels(df_results.index, rotation=20, ha='right', fontsize=10)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('F1-Score vs ROC-AUC', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart untuk semua metrik (grouped)
fig, ax = plt.subplots(figsize=(14, 5))

x     = np.arange(len(df_results.index))
n_met = len(metric_cols)
w     = 0.15
offsets = np.linspace(-(n_met-1)/2, (n_met-1)/2, n_met) * w

palette = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0']
for i, (metric, color) in enumerate(zip(metric_cols, palette)):
    vals = df_results[metric].values
    bars = ax.bar(x + offsets[i], vals, w, label=metric, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_results.index, rotation=15, ha='right', fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Perbandingan Semua Metrik — Student Dropout Classification',
             fontsize=13, fontweight='bold')
ax.legend(ncol=5, fontsize=10, loc='upper center', bbox_to_anchor=(0.5, 1.12))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Classification Report Detail & Kesimpulan

In [ ]:
for name, (_, y_pred, _) in model_objects.items():
    print("=" * 55)
    print(f"  {name}")
    print("=" * 55)
    print(classification_report(
        y_test, y_pred,
        target_names=['Tidak Dropout', 'Dropout']
    ))
    print()

In [ ]:
# ──────────────────────────────────────────────────────
# KESIMPULAN: Tentukan model terbaik
# ──────────────────────────────────────────────────────
best_f1  = df_results['F1-Score'].idxmax()
best_auc = df_results['ROC-AUC' ].idxmax()
best_acc = df_results['Accuracy'].idxmax()

print("=" * 60)
print("  KESIMPULAN PERBANDINGAN MODEL")
print("=" * 60)

print("\n📊 Tabel Ringkasan:")
print(df_results[metric_cols].round(4).to_string())

print(f"\n🏆 Model terbaik berdasarkan:")
print(f"   F1-Score  → {best_f1:<22} ({df_results.loc[best_f1,  'F1-Score']:.4f})")
print(f"   ROC-AUC   → {best_auc:<22} ({df_results.loc[best_auc, 'ROC-AUC']:.4f})")
print(f"   Accuracy  → {best_acc:<22} ({df_results.loc[best_acc, 'Accuracy']:.4f})")

# Ranking berdasarkan rata-rata semua metrik
df_results['Mean_Score'] = df_results[metric_cols].mean(axis=1)
best_overall = df_results['Mean_Score'].idxmax()

print(f"\n🥇 Model TERBAIK KESELURUHAN (rata-rata semua metrik):")
print(f"   ➜  {best_overall}")
print(f"   Mean Score: {df_results.loc[best_overall, 'Mean_Score']:.4f}")

print("\n📝 Ranking semua model (berdasarkan Mean Score):")
ranking = df_results['Mean_Score'].sort_values(ascending=False)
for i, (model_name, score) in enumerate(ranking.items(), 1):
    medal = ["🥇","🥈","🥉","  4.","  5."][i-1]
    print(f"   {medal} {model_name:<22} → {score:.4f}")

print("\n" + "=" * 60)